# 📊 Notebook 02: Model Results & Error Analysis
## Quantitative Evaluation & Ablation Ladder

This notebook evaluates the models on the test set:
1. **Pixel-Level Performance**: Dice, IoU, Precision, Recall with 1,000 image-level bootstrap 95% CIs.
2. **Instance-Level Evaluation**: Panoptic Quality (PQ), Segmentation Quality (SQ), Detection Quality (DQ).
3. **Morphological Agreement**: Nuclear count correlation, relative count error, mean area error.
4. **Error Classification & Visual Maps**: Categorization into Correct, Merged, Split, Missed, False Positive.
5. **AMP Hardware Benchmark**: FP32 vs FP16 inference latency and memory.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from metrics import compute_pixel_metrics, bootstrap_ci, panoptic_quality
from instances import binary_to_instances, match_instances, classify_errors, error_summary
from visualization import plot_error_map, plot_ablation_ladder

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
print("Evaluation modules loaded.")

### 1. Ablation Ladder (5 Configurations)
| Configuration | Dice [95% CI] | IoU [95% CI] | PQ | Count Error (↓) | Area Error (↓) |
|---|:---:|:---:|:---:|:---:|:---:|  
| M1: Scratch U-Net + BCE | - | - | - | - | - |
| M2: + Macenko Normalization | - | - | - | - | - |
| M3: + Augmentations | - | - | - | - | - |
| M4: + Combined BCE & Dice Loss | - | - | - | - | - |
| M5: + Pretrained ResNet-34 Encoder | - | - | - | - | - |

In [ ]:
# Synthetic demonstration of bootstrap CI calculation
np.random.seed(42)
pred_masks = [np.random.randint(0, 2, (256, 256)) for _ in range(15)]
gt_masks   = [np.random.randint(0, 2, (256, 256)) for _ in range(15)]

# Compute bootstrap 95% CI across test images
dice_mean, ci_low, ci_high = bootstrap_ci(
    metric_fn=lambda p, g: compute_pixel_metrics(p, g)["dice"],
    predictions=pred_masks,
    targets=gt_masks,
    n_bootstrap=500
)
print(f"Bootstrap Test Dice: {dice_mean:.4f} [95% CI: {ci_low:.4f} - {ci_high:.4f}]")

### 2. Error Categorization & Visual Error Maps
- **Green**: Correct (True Positive)
- **Red**: False Positive
- **Blue**: Missed (False Negative / Missed Nuclei)

In [ ]:
print("Error maps can be generated via plot_error_map(image, pred_instances, gt_instances)")